In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/sample_submission.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/data_dictionary.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/icd10_lookup.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/discharge_planning.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/procedures.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/diagnoses.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/prior_utilization.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/medications.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/hospitals.csv
/kaggle/input/competitions/machine-learning-fall-26-challenge-1-iba-karachi/index_encounters.csv
/kaggle/input/competitions/machine-learning-fall-26-chal

In [3]:

import re, gc
from pathlib import Path
import numpy as np
import pandas as pd

NOVA_ROOT = Path('/kaggle/input')
NOVA_FILES = {
    p.name.lower(): p
    for p in NOVA_ROOT.rglob('*.csv')
}

def nova_read(filename, wanted=None):
    p = NOVA_FILES.get(filename.lower())
    if p is None:
        print('NOT FOUND:', filename, flush=True)
        return None

    if wanted is None:
        d = pd.read_csv(p, low_memory=False)
    else:
        d = pd.read_csv(
            p,
            usecols=lambda c: c.lower() in wanted,
            low_memory=False
        )

    d.columns = d.columns.str.strip().str.lower()
    print('READ:', filename, d.shape, flush=True)
    return d

def nova_col(d, *names):
    if d is None:
        return None
    return next(
        (n for n in names if n in d.columns),
        None
    )

def nova_dt(s):
    try:
        return pd.to_datetime(
            s, errors='coerce', format='mixed'
        )
    except (TypeError, ValueError):
        return pd.to_datetime(s, errors='coerce')

def nova_safe(c):
    return not any(
        w in str(c).lower()
        for w in (
            'readmit', 'target', 'label',
            'outcome', 'future_', 'post_discharge',
            'next_admission', 'mortality',
            'date_of_death', 'death_date',
            'days_to_readm', 'days_until_readm',
            'readmission'
        )
    )

NOVA_train = nova_read('train.csv')
NOVA_test = nova_read('test.csv')

assert NOVA_train is not None and NOVA_test is not None
assert {
    'encounter_id', 'readmit_30d'
}.issubset(NOVA_train.columns)
assert 'encounter_id' in NOVA_test.columns

NOVA_y = NOVA_train[
    'readmit_30d'
].to_numpy(dtype=np.int8)

NOVA_n = len(NOVA_train)
NOVA_test_ids = NOVA_test['encounter_id'].copy()

NOVA_ids = pd.Index(
    pd.concat(
        [
            NOVA_train['encounter_id'],
            NOVA_test['encounter_id']
        ],
        ignore_index=True
    ).astype(str),
    name='encounter_id'
)

assert NOVA_ids.is_unique

NOVA_F = pd.DataFrame(index=NOVA_ids)
NOVA_AUDIT = []

def nova_add(frame, name):
    global NOVA_F

    if frame is None or frame.empty:
        print('EMPTY:', name, flush=True)
        return

    frame = frame.copy()
    frame.index = frame.index.astype(str)

    if not frame.index.is_unique:
        raise ValueError(
            'Non-unique encounter aggregate: ' + name
        )

    frame = frame.loc[
        :, ~frame.columns.duplicated()
    ]

    frame.columns = [
        re.sub('[^a-zA-Z0-9_]', '_', str(c))
        for c in frame.columns
    ]

    frame = frame.loc[
        :, ~frame.columns.isin(NOVA_F.columns)
    ]

    if not len(frame.columns):
        return

    coverage = int(
        NOVA_ids.isin(frame.index).sum()
    )

    NOVA_AUDIT.append((
        name, len(frame.columns), coverage
    ))

    NOVA_F = pd.concat(
        [
            NOVA_F,
            frame.reindex(NOVA_ids)
        ],
        axis=1
    )

    print(
        'ADDED', name, len(frame.columns),
        'columns, coverage', coverage,
        flush=True
    )

def nova_crosstab(d, key, field, prefix, top=30):
    if field is None or field not in d or d.empty:
        return

    v = (
        d[field].fillna('Missing')
        .astype(str).str.upper().str.strip()
    )

    top_vals = v.value_counts().head(top).index
    v = v.where(v.isin(top_vals), 'OTHER')

    tab = pd.crosstab(
        d[key].astype(str), v
    )

    tab.columns = [
        prefix + '_' + str(i)
        for i in range(len(tab.columns))
    ]

    nova_add(
        tab.astype(np.float32), prefix
    )

def nova_event(
    name, fields, time_names=(),
    required_time=False
):
    d = nova_read(name, fields)

    if d is None or 'encounter_id' not in d:
        return None

    d = d.dropna(
        subset=['encounter_id']
    ).copy()

    d['encounter_id'] = (
        d['encounter_id'].astype(str)
    )

    d = d[
        d['encounter_id'].isin(NOVA_ids)
    ]

    tc = nova_col(d, *time_names)

    if tc is not None:
        t = nova_dt(d[tc])

        admit = d['encounter_id'].map(
            NOVA_admit
        )

        discharge = d['encounter_id'].map(
            NOVA_discharge
        )

        good = (
            t.notna()
            & admit.notna()
            & discharge.notna()
            & t.ge(admit)
            & t.le(discharge)
        )

        print(
            'TIME-CHECK', name,
            'retained', int(good.sum()),
            'of', len(d),
            flush=True
        )

        d = d.loc[good].copy()
        d['_time'] = t.loc[good]
        d['_admit'] = admit.loc[good]
        d['_discharge'] = discharge.loc[good]

    elif required_time:
        print(
            'SKIP', name,
            ': no usable timestamp',
            flush=True
        )
        return None

    else:
        print(
            'ENCOUNTER-SCOPED (no timestamp):',
            name, flush=True
        )

    return d


# 1. Encounter, patient and hospital information

NOVA_idx = nova_read('index_encounters.csv')

assert (
    NOVA_idx is not None
    and 'encounter_id' in NOVA_idx
)

NOVA_idx['encounter_id'] = (
    NOVA_idx['encounter_id'].astype(str)
)

assert NOVA_idx['encounter_id'].is_unique

NOVA_idx = (
    NOVA_idx.set_index('encounter_id')
    .reindex(NOVA_ids)
)

NOVA_admit_col = nova_col(
    NOVA_idx,
    'admit_dt', 'admission_dt',
    'admit_date', 'admission_date'
)

NOVA_discharge_col = nova_col(
    NOVA_idx,
    'discharge_dt', 'discharge_date',
    'discharged_dt'
)

assert NOVA_admit_col and NOVA_discharge_col, (
    'Admission/discharge date columns not found'
)

NOVA_admit = nova_dt(
    NOVA_idx[NOVA_admit_col]
)
NOVA_admit.index = NOVA_ids

NOVA_discharge = nova_dt(
    NOVA_idx[NOVA_discharge_col]
)
NOVA_discharge.index = NOVA_ids

NOVA_F['nova_stay_days'] = (
    NOVA_discharge - NOVA_admit
).dt.total_seconds() / 86400

NOVA_F['nova_admit_month'] = (
    NOVA_admit.dt.month
)
NOVA_F['nova_admit_weekday'] = (
    NOVA_admit.dt.dayofweek
)
NOVA_F['nova_discharge_weekday'] = (
    NOVA_discharge.dt.dayofweek
)
NOVA_F['nova_discharge_hour'] = (
    NOVA_discharge.dt.hour
)
NOVA_F['nova_weekend_discharge'] = (
    NOVA_discharge.dt.dayofweek >= 5
).astype('float32')

NOVA_hosp_groups = (
    NOVA_idx['hospital_id']
    .iloc[:NOVA_n]
    .reset_index(drop=True)
    if 'hospital_id' in NOVA_idx
    else None
)

NOVA_pid = (
    NOVA_idx['patient_id'].astype(str)
    if 'patient_id' in NOVA_idx
    else None
)

for c in NOVA_idx:
    if (
        c in (
            NOVA_admit_col,
            NOVA_discharge_col,
            'patient_id'
        )
        or not nova_safe(c)
    ):
        continue

    if (
        'date' in c
        or c.endswith('_dt')
        or 'timestamp' in c
    ):
        continue

    NOVA_F['nova_index_' + c] = (
        NOVA_idx[c].to_numpy()
    )

for fname, joincol, prefix in [
    ('patients.csv', 'patient_id', 'patient'),
    ('hospitals.csv', 'hospital_id', 'hospital')
]:
    d = nova_read(fname)

    if (
        d is None
        or joincol not in d
        or joincol not in NOVA_idx
    ):
        continue

    d[joincol] = d[joincol].astype(str)

    d = (
        d.drop_duplicates(joincol)
        .set_index(joincol)
    )

    for c in d.columns:
        if (
            not nova_safe(c)
            or 'date' in c
            or c.endswith('_dt')
            or 'timestamp' in c
            or 'birth' in c
            or c == 'dob'
        ):
            continue

        NOVA_F[
            'nova_' + prefix + '_' + c
        ] = (
            NOVA_idx[joincol]
            .astype(str)
            .map(d[c])
            .to_numpy()
        )

    if prefix == 'patient':
        dob = nova_col(
            d, 'dob', 'birth_date',
            'date_of_birth'
        )

        if dob:
            birth = nova_dt(
                NOVA_idx[joincol]
                .astype(str)
                .map(d[dob])
            )

            age = (
                NOVA_admit - birth
            ).dt.days / 365.25

            NOVA_F['nova_age_from_birth'] = (
                age.where(age.between(0, 115))
            )

    del d


# 2. Diagnosis codes and clinical conditions

NOVA_d = nova_event(
    'diagnoses.csv',
    {
        'encounter_id', 'icd10_code',
        'icd_10_code', 'diagnosis_code',
        'code', 'diagnosis_type', 'is_primary'
    }
)

if NOVA_d is not None and len(NOVA_d):
    g = NOVA_d.groupby('encounter_id')

    nova_add(
        g.size().to_frame('nova_dx_count'),
        'diagnoses_count'
    )

    c = nova_col(
        NOVA_d, 'icd10_code',
        'icd_10_code', 'diagnosis_code', 'code'
    )

    if c:
        NOVA_d['_code'] = (
            NOVA_d[c].fillna('UNK')
            .astype(str).str.upper()
            .str.replace('.', '', regex=False)
        )

        NOVA_d['_three'] = (
            NOVA_d['_code'].str[:3]
        )

        NOVA_d['_chapter'] = (
            NOVA_d['_code'].str[:1]
        )

        nova_add(
            g['_three'].nunique().to_frame(
                'nova_dx_unique'
            ),
            'diagnoses_unique'
        )

        nova_crosstab(
            NOVA_d, 'encounter_id',
            '_three', 'nova_dx3', 65
        )

        nova_crosstab(
            NOVA_d, 'encounter_id',
            '_chapter', 'nova_dxchapter', 26
        )

        for label, regex in [
            ('diabetes', r'^E1[0-4]'),
            ('heart_failure', r'^I50'),
            ('kidney', r'^N18'),
            ('copd', r'^J44'),
            ('malignancy', r'^C'),
            ('sepsis', r'^A4[01]')
        ]:
            f = NOVA_d['_three'].str.contains(
                regex, regex=True, na=False
            )

            nova_add(
                f.groupby(
                    NOVA_d.encounter_id
                ).max().astype('float32').to_frame(
                    'nova_dx_' + label
                ),
                'dx_' + label
            )

    del NOVA_d


# 3. Medications (fixes V10's lost records)

NOVA_d = nova_event(
    'medications.csv',
    {
        'encounter_id', 'drug_class',
        'med_class', 'medication_class',
        'drug_name', 'medication_name',
        'continued_at_discharge'
    }
)

if NOVA_d is not None and len(NOVA_d):
    g = NOVA_d.groupby('encounter_id')

    nova_add(
        g.size().to_frame('nova_med_count'),
        'med_count'
    )

    c = nova_col(
        NOVA_d, 'drug_class',
        'med_class', 'medication_class'
    )

    if c:
        nova_add(
            g[c].nunique().to_frame(
                'nova_med_unique_classes'
            ),
            'med_classes'
        )

        nova_crosstab(
            NOVA_d, 'encounter_id',
            c, 'nova_medclass', 45
        )

    c = nova_col(
        NOVA_d, 'drug_name',
        'medication_name'
    )

    if c:
        nova_crosstab(
            NOVA_d, 'encounter_id',
            c, 'nova_medname', 18
        )

    c = nova_col(
        NOVA_d, 'continued_at_discharge'
    )

    if c:
        z = (
            NOVA_d[c].fillna('')
            .astype(str).str.lower()
            .isin(['1', 'true', 'yes', 'y'])
        )

        nova_add(
            z.groupby(
                NOVA_d.encounter_id
            ).sum().to_frame(
                'nova_med_continued'
            ),
            'med_discharge'
        )

    del NOVA_d


# 4. Procedures and unit transfers

for fname, pref, options, top in [
    (
        'procedures.csv', 'nova_proc',
        [
            'procedure_code',
            'procedure_type',
            'procedure_name', 'code'
        ],
        30
    ),
    (
        'unit_transfers.csv', 'nova_transfer',
        [
            'unit_type', 'to_unit',
            'unit_name', 'care_unit'
        ],
        18
    )
]:
    NOVA_d = nova_event(
        fname, {'encounter_id', *options}
    )

    if NOVA_d is None or NOVA_d.empty:
        continue

    nova_add(
        NOVA_d.groupby(
            'encounter_id'
        ).size().to_frame(pref + '_count'),
        pref + '_count'
    )

    c = nova_col(NOVA_d, *options)

    if c:
        nova_crosstab(
            NOVA_d, 'encounter_id',
            c, pref, top
        )

        if 'transfer' in pref:
            icu = (
                NOVA_d[c].fillna('')
                .astype(str)
                .str.contains(
                    'ICU|INTENSIVE|CRITICAL',
                    case=False, regex=True
                )
            )

            nova_add(
                icu.groupby(
                    NOVA_d.encounter_id
                ).sum().to_frame(
                    'nova_icu_count'
                ),
                'icu_count'
            )

    del NOVA_d


# 5. Discharge planning

NOVA_d = nova_read(
    'discharge_planning.csv'
)

if (
    NOVA_d is not None
    and 'encounter_id' in NOVA_d
):
    NOVA_d['encounter_id'] = (
        NOVA_d.encounter_id.astype(str)
    )

    NOVA_d = NOVA_d[
        NOVA_d.encounter_id.isin(NOVA_ids)
    ]

    nova_add(
        NOVA_d.groupby(
            'encounter_id'
        ).size().to_frame(
            'nova_plan_count'
        ),
        'plan_count'
    )

    plan_cols = [
        c for c in NOVA_d.columns
        if c != 'encounter_id'
        and nova_safe(c)
        and not any(
            w in c
            for w in [
                'date', '_dt', 'time',
                'note', 'text',
                'description', '_id'
            ]
        )
    ]

    for c in plan_cols[:12]:
        if NOVA_d[c].nunique(
            dropna=True
        ) <= 25:
            nova_crosstab(
                NOVA_d, 'encounter_id',
                c, 'nova_plan_' + c, 12
            )

    del NOVA_d


# 6. Historical utilization before admission

NOVA_d = nova_read(
    'prior_utilization.csv',
    {
        'patient_id', 'visit_dt',
        'visit_date', 'visit_type',
        'encounter_type', 'care_setting'
    }
)

if (
    NOVA_d is not None
    and NOVA_pid is not None
    and 'patient_id' in NOVA_d
):
    c = nova_col(
        NOVA_d, 'visit_dt', 'visit_date'
    )

    if c:
        NOVA_d['patient_id'] = (
            NOVA_d.patient_id.astype(str)
        )

        NOVA_d['_visit'] = nova_dt(
            NOVA_d[c]
        )

        NOVA_map = pd.DataFrame({
            'encounter_id': NOVA_ids.to_numpy(),
            'patient_id': NOVA_pid.to_numpy(),
            'admit': NOVA_admit.to_numpy()
        })

        NOVA_d = NOVA_d.merge(
            NOVA_map,
            on='patient_id',
            how='inner',
            validate='many_to_many'
        )

        NOVA_d['_days'] = (
            NOVA_d.admit - NOVA_d._visit
        ).dt.total_seconds() / 86400

        NOVA_d = NOVA_d[
            NOVA_d._days.gt(0)
            & NOVA_d._days.le(3650)
        ]

        g = NOVA_d.groupby('encounter_id')

        f = pd.DataFrame({
            'nova_prior_count': g.size(),
            'nova_prior_recency_days': g._days.min()
        })

        for days in (
            7, 30, 90, 180, 365, 730
        ):
            f[
                'nova_prior_' + str(days) + 'd'
            ] = (
                NOVA_d._days.le(days)
                .groupby(NOVA_d.encounter_id)
                .sum()
            )

        nova_add(f, 'prior_visits')

        c = nova_col(
            NOVA_d, 'visit_type',
            'encounter_type', 'care_setting'
        )

        if c:
            nova_crosstab(
                NOVA_d, 'encounter_id',
                c, 'nova_prior_type', 12
            )

        del NOVA_map

    del NOVA_d


# 7. Laboratory results and abnormality trends

NOVA_d = nova_event(
    'lab_results.csv',
    {
        'encounter_id', 'collected_dt',
        'collection_dt', 'result_dt',
        'lab_dt', 'charttime',
        'lab_name', 'test_name',
        'lab_test', 'analyte',
        'value', 'result_value',
        'numeric_value', 'lab_value',
        'result', 'abnormal_flag',
        'flag', 'abnormality'
    },
    (
        'collected_dt', 'collection_dt',
        'result_dt', 'lab_dt', 'charttime'
    ),
    required_time=True
)

if NOVA_d is not None and len(NOVA_d):
    g = NOVA_d.groupby('encounter_id')

    nova_add(
        g.size().to_frame(
            'nova_lab_count'
        ),
        'lab_count'
    )

    c = nova_col(
        NOVA_d,
        'abnormal_flag', 'flag',
        'abnormality'
    )

    if c:
        flag = (
            NOVA_d[c].fillna('')
            .astype(str).str.upper().str.strip()
        )

        bad = flag.isin([
            'H', 'L', 'HH', 'LL',
            'HIGH', 'LOW',
            'ABNORMAL', 'ABN', 'A',
            'CRITICAL', 'CRIT'
        ])

        f = pd.DataFrame({
            'nova_lab_abnormal_count':
                bad.groupby(
                    NOVA_d.encounter_id
                ).sum(),
            'nova_lab_abnormal_rate':
                bad.groupby(
                    NOVA_d.encounter_id
                ).mean()
        })

        nova_add(f, 'lab_abnormal')

        last24 = (
            NOVA_d._discharge - NOVA_d._time
        ).dt.total_seconds().between(
            0, 86400
        )

        f2 = pd.DataFrame({
            'nova_lab_last24_count':
                last24.groupby(
                    NOVA_d.encounter_id
                ).sum(),
            'nova_lab_last24_abnormal':
                (last24 & bad).groupby(
                    NOVA_d.encounter_id
                ).sum()
        })

        nova_add(f2, 'lab_last24')

    name = nova_col(
        NOVA_d,
        'lab_name', 'test_name',
        'lab_test', 'analyte'
    )

    value = nova_col(
        NOVA_d,
        'value', 'result_value',
        'numeric_value', 'lab_value',
        'result'
    )

    if name:
        nova_crosstab(
            NOVA_d, 'encounter_id',
            name, 'nova_labtest', 28
        )

    if name and value:
        NOVA_d['_num'] = pd.to_numeric(
            NOVA_d[value], errors='coerce'
        )

        NOVA_d['_kind'] = (
            NOVA_d[name].fillna('Unknown')
            .astype(str).str.upper().str.strip()
        )

        top = (
            NOVA_d['_kind']
            .value_counts()
            .head(20).index
        )

        sm = NOVA_d[
            NOVA_d._kind.isin(top)
            & NOVA_d._num.notna()
        ].sort_values('_time')

        if len(sm):
            stat = sm.groupby(
                ['encounter_id', '_kind']
            )['_num'].agg([
                'mean', 'std', 'min',
                'max', 'first', 'last'
            ])

            stat['delta'] = (
                stat['last'] - stat['first']
            )

            wide = stat.unstack('_kind')

            wide.columns = [
                'nova_labnum_' + str(i)
                for i in range(wide.shape[1])
            ]

            nova_add(
                wide, 'lab_values'
            )

        del sm

    del NOVA_d
    gc.collect()


# 8. Vital signs and clinical trajectories

NOVA_d = nova_event(
    'vitals.csv',
    {
        'encounter_id', 'charttime',
        'recorded_dt', 'vital_dt',
        'event_dt', 'vital_type',
        'vital_name', 'measurement_type',
        'value', 'vital_value',
        'numeric_value'
    },
    (
        'charttime', 'recorded_dt',
        'vital_dt', 'event_dt'
    ),
    required_time=True
)

if NOVA_d is not None and len(NOVA_d):
    name = nova_col(
        NOVA_d,
        'vital_type', 'vital_name',
        'measurement_type'
    )

    value = nova_col(
        NOVA_d,
        'value', 'vital_value',
        'numeric_value'
    )

    if name and value:
        NOVA_d['_kind'] = (
            NOVA_d[name].fillna('UNK')
            .astype(str).str.upper().str.strip()
        )

        NOVA_d['_num'] = pd.to_numeric(
            NOVA_d[value], errors='coerce'
        )

        temp = (
            NOVA_d['_kind'].isin([
                'TEMP', 'TEMPERATURE'
            ])
            & NOVA_d._num.between(45, 115)
        )

        NOVA_d.loc[temp, '_num'] = (
            NOVA_d.loc[temp, '_num'] - 32
        ) * 5 / 9

        NOVA_d = NOVA_d.sort_values('_time')

        top = (
            NOVA_d['_kind']
            .value_counts()
            .head(10).index
        )

        NOVA_d = NOVA_d[
            NOVA_d._kind.isin(top)
        ]

        stat = NOVA_d.groupby(
            ['encounter_id', '_kind']
        )['_num'].agg([
            'mean', 'std', 'min',
            'max', 'first', 'last', 'count'
        ])

        stat['delta'] = (
            stat['last'] - stat['first']
        )

        wide = stat.unstack('_kind')

        wide.columns = [
            'nova_vital_' + str(i)
            for i in range(wide.shape[1])
        ]

        nova_add(
            wide, 'vital_stats'
        )

        last24 = (
            NOVA_d._discharge - NOVA_d._time
        ).dt.total_seconds().between(
            0, 86400
        )

        last = (
            NOVA_d[last24]
            .groupby(
                ['encounter_id', '_kind']
            )['_num']
            .mean()
            .unstack('_kind')
        )

        last.columns = [
            'nova_vital_last24_' + str(i)
            for i in range(last.shape[1])
        ]

        nova_add(
            last, 'vital_last24'
        )

    del NOVA_d
    gc.collect()


# 9. Cross-table clinical interactions

for a, b, c in [
    (
        'nova_lab_abnormal_count',
        'nova_lab_count',
        'nova_abnormal_fraction'
    ),
    (
        'nova_med_count',
        'nova_stay_days',
        'nova_med_per_day'
    ),
    (
        'nova_dx_count',
        'nova_stay_days',
        'nova_dx_per_day'
    ),
    (
        'nova_transfer_count',
        'nova_stay_days',
        'nova_transfers_per_day'
    ),
    (
        'nova_prior_30d',
        'nova_prior_365d',
        'nova_prior_recent_share'
    )
]:
    if a in NOVA_F and b in NOVA_F:
        NOVA_F[c] = (
            pd.to_numeric(
                NOVA_F[a], errors='coerce'
            )
            /
            (
                1 + pd.to_numeric(
                    NOVA_F[b], errors='coerce'
                )
            )
        )


# 10. Final master dataset

for c in list(NOVA_F.columns):
    if (
        not nova_safe(c)
        or pd.api.types.is_datetime64_any_dtype(
            NOVA_F[c]
        )
    ):
        NOVA_F.drop(
            columns=c, inplace=True
        )

    elif NOVA_F[c].dtype == 'object':
        NOVA_F[c] = (
            NOVA_F[c].fillna('Missing')
            .astype(str)
        )

        if (
            NOVA_F[c].nunique() > 200
            or NOVA_F[c].nunique()
            > len(NOVA_F) * 0.2
        ):
            NOVA_F.drop(
                columns=c, inplace=True
            )

NOVA_F = NOVA_F.replace(
    [np.inf, -np.inf], np.nan
)

NOVA_F['nova_missing_count'] = (
    NOVA_F.isna()
    .sum(axis=1)
    .astype(np.float32)
)

NOVA_F = NOVA_F.loc[
    :, ~NOVA_F.columns.duplicated()
]

NOVA_X = (
    NOVA_F.iloc[:NOVA_n]
    .reset_index(drop=True)
)

NOVA_T = (
    NOVA_F.iloc[NOVA_n:]
    .reset_index(drop=True)
)

assert len(NOVA_X) == len(NOVA_train)
assert len(NOVA_T) == len(NOVA_test)
assert NOVA_X.columns.equals(NOVA_T.columns)

print(
    '\nNOVA DATA READY:',
    NOVA_X.shape,
    NOVA_T.shape,
    flush=True
)

print(
    pd.DataFrame(
        NOVA_AUDIT,
        columns=[
            'source',
            'features',
            'matched_encounters'
        ]
    ).to_string(index=False)
)


READ: train.csv (76975, 2)
READ: test.csv (32812, 1)
READ: index_encounters.csv (109787, 12)
READ: patients.csv (109787, 9)
READ: hospitals.csv (40, 5)
READ: diagnoses.csv (451976, 3)
ENCOUNTER-SCOPED (no timestamp): diagnoses.csv
ADDED diagnoses_count 1 columns, coverage 109787
ADDED diagnoses_unique 1 columns, coverage 109787
ADDED nova_dx3 63 columns, coverage 109787
ADDED nova_dxchapter 16 columns, coverage 109787
ADDED dx_diabetes 1 columns, coverage 109787
ADDED dx_heart_failure 1 columns, coverage 109787
ADDED dx_kidney 1 columns, coverage 109787
ADDED dx_copd 1 columns, coverage 109787
ADDED dx_malignancy 1 columns, coverage 109787
ADDED dx_sepsis 1 columns, coverage 109787
READ: medications.csv (447278, 4)
ENCOUNTER-SCOPED (no timestamp): medications.csv
ADDED med_count 1 columns, coverage 107628
ADDED med_classes 1 columns, coverage 107628
ADDED nova_medclass 22 columns, coverage 107628
ADDED nova_medname 19 columns, coverage 107628
ADDED med_discharge 1 columns, coverage 107

In [4]:

import gc
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

assert (
    'NOVA_X' in globals()
    and 'NOVA_T' in globals()
), 'Run NOVA Cell 1 first'

NOVA_X2 = NOVA_X.copy()
NOVA_T2 = NOVA_T.copy()

# Remove constant and almost-empty features

NOVA_keep = [
    c for c in NOVA_X2.columns
    if (
        NOVA_X2[c].notna().sum()
        >= max(25, int(.001 * len(NOVA_X2)))
        and NOVA_X2[c].nunique(
            dropna=True
        ) > 1
    )
]

NOVA_X2 = NOVA_X2[NOVA_keep]
NOVA_T2 = NOVA_T2[NOVA_keep]

NOVA_cat = NOVA_X2.select_dtypes(
    exclude='number'
).columns.tolist()

NOVA_num = [
    c for c in NOVA_X2.columns
    if c not in NOVA_cat
]

for c in NOVA_cat:
    vals = pd.concat(
        [
            NOVA_X2[c],
            NOVA_T2[c]
        ],
        ignore_index=True
    ).fillna('Missing').astype(str)

    levels = pd.Categorical(
        vals
    ).categories

    NOVA_X2[c] = pd.Categorical(
        NOVA_X2[c].fillna('Missing')
        .astype(str),
        categories=levels
    )

    NOVA_T2[c] = pd.Categorical(
        NOVA_T2[c].fillna('Missing')
        .astype(str),
        categories=levels
    )

for c in NOVA_num:
    NOVA_X2[c] = pd.to_numeric(
        NOVA_X2[c], errors='coerce'
    ).astype(np.float32)

    NOVA_T2[c] = pd.to_numeric(
        NOVA_T2[c], errors='coerce'
    ).astype(np.float32)

NOVA_folds = list(
    StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ).split(NOVA_X2, NOVA_y)
)

NOVA_oof = {}
NOVA_pred = {}
NOVA_fold_report = []

# Two decision-tree boosting configurations

NOVA_models = {
    'TREE_SMALL': dict(
        num_leaves=9,
        min_child_samples=120,
        reg_lambda=18,
        reg_alpha=1.0,
        max_depth=5
    ),
    'TREE_MEDIUM': dict(
        num_leaves=19,
        min_child_samples=75,
        reg_lambda=12,
        reg_alpha=0.6,
        max_depth=-1
    )
}

for label, params in NOVA_models.items():
    NOVA_oof[label] = np.zeros(
        len(NOVA_y)
    )

    NOVA_pred[label] = np.zeros(
        len(NOVA_T2)
    )

    for k, (tr, va) in enumerate(
        NOVA_folds, 1
    ):
        print(
            'NOVA', label,
            'FOLD', k, '/3',
            flush=True
        )

        m = LGBMClassifier(
            n_estimators=1200,
            learning_rate=0.035,
            objective='binary',
            colsample_bytree=0.82,
            subsample=0.85,
            subsample_freq=1,
            random_state=500 + k,
            n_jobs=4,
            verbosity=-1,
            **params
        )

        m.fit(
            NOVA_X2.iloc[tr],
            NOVA_y[tr],
            eval_set=[
                (
                    NOVA_X2.iloc[va],
                    NOVA_y[va]
                )
            ],
            eval_metric='auc',
            callbacks=[
                early_stopping(
                    95, verbose=False
                ),
                log_evaluation(0)
            ]
        )

        NOVA_oof[label][va] = (
            m.predict_proba(
                NOVA_X2.iloc[va]
            )[:, 1]
        )

        NOVA_pred[label] += (
            m.predict_proba(
                NOVA_T2
            )[:, 1] / 3
        )

        auc = roc_auc_score(
            NOVA_y[va],
            NOVA_oof[label][va]
        )

        trainauc = roc_auc_score(
            NOVA_y[tr],
            m.predict_proba(
                NOVA_X2.iloc[tr]
            )[:, 1]
        )

        NOVA_fold_report.append((
            label,
            k,
            auc,
            trainauc,
            trainauc - auc,
            m.best_iteration_
        ))

        print(
            'AUC:', round(auc, 6),
            'TRAIN-VALID GAP:',
            round(trainauc - auc, 5),
            flush=True
        )

        del m
        gc.collect()


# Prepare sparse data for the linear model

NOVA_median = (
    NOVA_X2[NOVA_num]
    .median()
    .fillna(0)
)

NOVA_lo = NOVA_X2[
    NOVA_num
].quantile(.05)

NOVA_hi = NOVA_X2[
    NOVA_num
].quantile(.95)

NOVA_scale = (
    (NOVA_hi - NOVA_lo)
    .replace(0, 1)
    .fillna(1)
)

def nova_numeric_matrix(df):
    a = (
        df[NOVA_num].fillna(
            NOVA_median
        ) - NOVA_median
    ) / NOVA_scale

    return csr_matrix(
        np.clip(
            a.to_numpy(
                dtype=np.float32
            ),
            -10, 10
        )
    )

NOVA_linear_x = nova_numeric_matrix(
    NOVA_X2
)

NOVA_linear_t = nova_numeric_matrix(
    NOVA_T2
)

if NOVA_cat:
    enc = OneHotEncoder(
        handle_unknown='ignore',
        min_frequency=20,
        dtype=np.float32
    )

    both = pd.concat(
        [
            NOVA_X2[NOVA_cat],
            NOVA_T2[NOVA_cat]
        ],
        ignore_index=True
    ).astype(str)

    enc.fit(both)

    NOVA_linear_x = hstack(
        [
            NOVA_linear_x,
            enc.transform(
                NOVA_X2[NOVA_cat].astype(str)
            )
        ],
        format='csr'
    )

    NOVA_linear_t = hstack(
        [
            NOVA_linear_t,
            enc.transform(
                NOVA_T2[NOVA_cat].astype(str)
            )
        ],
        format='csr'
    )

    del both, enc


# Classroom Decision Tree with entropy

tr0, va0 = NOVA_folds[0]

NOVA_dt_model = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=6,
    min_samples_leaf=100,
    random_state=42
)

NOVA_dt_model.fit(
    NOVA_linear_x[tr0],
    NOVA_y[tr0]
)

print(
    'NOVA ENTROPY DECISION TREE (fold 1):',
    round(
        roc_auc_score(
            NOVA_y[va0],
            NOVA_dt_model.predict_proba(
                NOVA_linear_x[va0]
            )[:, 1]
        ),
        6
    ),
    flush=True
)

del NOVA_dt_model


# Regularized linear classifier

NOVA_oof['LINEAR_L2'] = np.zeros(
    len(NOVA_y)
)

NOVA_pred['LINEAR_L2'] = np.zeros(
    len(NOVA_T2)
)

for k, (tr, va) in enumerate(
    NOVA_folds, 1
):
    print(
        'NOVA LINEAR_L2 FOLD',
        k, '/3',
        flush=True
    )

    lr = LogisticRegression(
        C=0.12,
        solver='liblinear',
        max_iter=250,
        random_state=42
    )

    lr.fit(
        NOVA_linear_x[tr],
        NOVA_y[tr]
    )

    NOVA_oof['LINEAR_L2'][va] = (
        lr.predict_proba(
            NOVA_linear_x[va]
        )[:, 1]
    )

    NOVA_pred['LINEAR_L2'] += (
        lr.predict_proba(
            NOVA_linear_t
        )[:, 1] / 3
    )

    valauc = roc_auc_score(
        NOVA_y[va],
        NOVA_oof['LINEAR_L2'][va]
    )

    trainauc = roc_auc_score(
        NOVA_y[tr],
        lr.predict_proba(
            NOVA_linear_x[tr]
        )[:, 1]
    )

    NOVA_fold_report.append((
        'LINEAR_L2',
        k,
        valauc,
        trainauc,
        trainauc - valauc,
        None
    ))

    print(
        'AUC:', round(valauc, 6),
        flush=True
    )

    del lr
    gc.collect()


# Compare independent models and ensembles

NOVA_candidates = {
    'TREE_SMALL': ('TREE_SMALL',),
    'TREE_MEDIUM': ('TREE_MEDIUM',),
    'LINEAR_L2': ('LINEAR_L2',),
    'TREE_50_50': (
        'TREE_SMALL',
        'TREE_MEDIUM'
    ),
    'TREE_80_LINEAR20': (
        'TREE_SMALL',
        'TREE_MEDIUM',
        'LINEAR_L2'
    ),
    'TREE_60_LINEAR40': (
        'TREE_SMALL',
        'TREE_MEDIUM',
        'LINEAR_L2',
        'LINEAR_L2'
    )
}

def nova_blend(name, predictions):
    parts = NOVA_candidates[name]

    if name == 'TREE_80_LINEAR20':
        return (
            .4 * predictions['TREE_SMALL']
            + .4 * predictions['TREE_MEDIUM']
            + .2 * predictions['LINEAR_L2']
        )

    if name == 'TREE_60_LINEAR40':
        return (
            .3 * predictions['TREE_SMALL']
            + .3 * predictions['TREE_MEDIUM']
            + .4 * predictions['LINEAR_L2']
        )

    return np.mean(
        [
            predictions[p]
            for p in parts
        ],
        axis=0
    )

NOVA_scores = []

for name in NOVA_candidates:
    p = nova_blend(
        name, NOVA_oof
    )

    NOVA_scores.append((
        roc_auc_score(
            NOVA_y, p
        ),
        average_precision_score(
            NOVA_y, p
        ),
        name
    ))

NOVA_scores.sort(reverse=True)

print(
    '\nNOVA MODEL COMPARISON (3-fold OOF):',
    flush=True
)

for auc, ap, name in NOVA_scores:
    print(
        f'{name:24s} '
        f'ROC-AUC={auc:.6f} '
        f'PR-AUC={ap:.6f}',
        flush=True
    )

print(
    '\nNOVA FOLD REPORT:',
    flush=True
)

print(
    pd.DataFrame(
        NOVA_fold_report,
        columns=[
            'model', 'fold',
            'validation_auc',
            'training_auc',
            'overfit_gap',
            'best_iteration'
        ]
    ).round(5).to_string(index=False)
)

print(
    '\nV9 historical OOF: 0.748383; '
    'public: 0.74214',
    flush=True
)


NOVA TREE_SMALL FOLD 1 /3
AUC: 0.744641 TRAIN-VALID GAP: 0.04321
NOVA TREE_SMALL FOLD 2 /3
AUC: 0.74513 TRAIN-VALID GAP: 0.04586
NOVA TREE_SMALL FOLD 3 /3
AUC: 0.736415 TRAIN-VALID GAP: 0.05971
NOVA TREE_MEDIUM FOLD 1 /3
AUC: 0.742064 TRAIN-VALID GAP: 0.06514
NOVA TREE_MEDIUM FOLD 2 /3
AUC: 0.744227 TRAIN-VALID GAP: 0.05369
NOVA TREE_MEDIUM FOLD 3 /3
AUC: 0.735253 TRAIN-VALID GAP: 0.08134
NOVA ENTROPY DECISION TREE (fold 1): 0.698004
NOVA LINEAR_L2 FOLD 1 /3
AUC: 0.746447
NOVA LINEAR_L2 FOLD 2 /3
AUC: 0.745235
NOVA LINEAR_L2 FOLD 3 /3
AUC: 0.73395

NOVA MODEL COMPARISON (3-fold OOF):
TREE_60_LINEAR40         ROC-AUC=0.745043 PR-AUC=0.315529
TREE_80_LINEAR20         ROC-AUC=0.743990 PR-AUC=0.313729
TREE_SMALL               ROC-AUC=0.741985 PR-AUC=0.310155
LINEAR_L2                ROC-AUC=0.741778 PR-AUC=0.308585
TREE_50_50               ROC-AUC=0.741690 PR-AUC=0.309680
TREE_MEDIUM              ROC-AUC=0.740306 PR-AUC=0.307214

NOVA FOLD REPORT:
      model  fold  validation_auc  trainin

In [5]:

import base64
from IPython.display import HTML, display
from sklearn.model_selection import GroupShuffleSplit

NOVA_best_auc, NOVA_best_ap, NOVA_best_name = (
    NOVA_scores[0]
)

print(
    'NOVA WINNER:',
    NOVA_best_name,
    'CV ROC-AUC:',
    round(NOVA_best_auc, 6),
    flush=True
)

def nova_stress(title, tr, va):
    if (
        len(tr) < 1000
        or len(va) < 500
        or len(np.unique(NOVA_y[va])) < 2
    ):
        print(
            title, ': insufficient samples'
        )
        return

    m = LGBMClassifier(
        n_estimators=220,
        learning_rate=0.035,
        num_leaves=9,
        min_child_samples=120,
        reg_lambda=18,
        reg_alpha=1,
        colsample_bytree=.82,
        subsample=.85,
        subsample_freq=1,
        random_state=42,
        n_jobs=4,
        verbosity=-1
    )

    m.fit(
        NOVA_X2.iloc[tr],
        NOVA_y[tr]
    )

    trainauc = roc_auc_score(
        NOVA_y[tr],
        m.predict_proba(
            NOVA_X2.iloc[tr]
        )[:, 1]
    )

    valauc = roc_auc_score(
        NOVA_y[va],
        m.predict_proba(
            NOVA_X2.iloc[va]
        )[:, 1]
    )

    print(
        f'{title}: '
        f'train={trainauc:.5f} '
        f'holdout={valauc:.5f} '
        f'gap={trainauc-valauc:.5f} '
        f'n={len(va)}',
        flush=True
    )

print(
    '\nNOVA GENERALIZATION CHECKS:',
    flush=True
)

# Later discharge dates

NOVA_end = (
    NOVA_discharge.iloc[:NOVA_n]
    .reset_index(drop=True)
)

valid = NOVA_end.notna().to_numpy()

if (
    valid.sum() >= 10000
    and NOVA_end[valid].nunique() > 20
):
    order = np.flatnonzero(valid)[
        np.argsort(
            NOVA_end[valid].to_numpy()
        )
    ]

    cut = int(len(order) * .8)

    nova_stress(
        'LATER DISCHARGES',
        order[:cut],
        order[cut:]
    )
else:
    print(
        'LATER DISCHARGES: unavailable'
    )

# Hospitals not seen during training

if (
    NOVA_hosp_groups is not None
    and NOVA_hosp_groups.nunique() > 5
):
    groups = (
        NOVA_hosp_groups.fillna('Missing')
        .astype(str).to_numpy()
    )

    tr, va = next(
        GroupShuffleSplit(
            n_splits=1,
            test_size=.2,
            random_state=42
        ).split(
            NOVA_X2,
            NOVA_y,
            groups=groups
        )
    )

    nova_stress(
        'UNSEEN HOSPITALS',
        tr, va
    )
else:
    print(
        'UNSEEN HOSPITALS: unavailable'
    )


# Save two independently named candidates

NOVA_tree_name = max(
    (
        t for t in NOVA_scores
        if 'LINEAR' not in t[2]
    ),
    key=lambda t: t[0]
)[2]

NOVA_exports = [
    ('NOVA_BEST', NOVA_best_name),
    ('NOVA_TREE_ONLY', NOVA_tree_name)
]

for filename, name in NOVA_exports:
    pred = np.asarray(
        nova_blend(
            name, NOVA_pred
        )
    )

    assert len(pred) == len(NOVA_test_ids)
    assert np.isfinite(pred).all()
    assert ((pred >= 0) & (pred <= 1)).all()

    out = pd.DataFrame({
        'encounter_id':
            NOVA_test_ids.to_numpy(),
        'readmit_30d': pred
    })

    assert out.encounter_id.is_unique

    path = (
        '/kaggle/working/'
        + filename + '.csv'
    )

    out.to_csv(
        path, index=False
    )

    print(
        'SAVED:', path,
        'candidate:', name,
        flush=True
    )

    with open(path, 'rb') as f:
        encoded = base64.b64encode(
            f.read()
        ).decode('ascii')

    display(
        HTML(
            '<a download="'
            + filename + '.csv" '
            + 'href="data:text/csv;base64,'
            + encoded
            + '">DOWNLOAD '
            + filename + '.csv</a>'
        )
    )

pd.DataFrame(
    NOVA_AUDIT,
    columns=[
        'source',
        'features',
        'matched_encounters'
    ]
).to_csv(
    '/kaggle/working/NOVA_SOURCE_AUDIT.csv',
    index=False
)

pd.DataFrame(
    NOVA_fold_report,
    columns=[
        'model', 'fold',
        'validation_auc',
        'training_auc',
        'overfit_gap',
        'best_iteration'
    ]
).to_csv(
    '/kaggle/working/NOVA_FOLD_REPORT.csv',
    index=False
)

print(
    '\nHistorical V9 OOF: 0.748383; '
    'public: 0.74214. '
    'Review results before submitting.',
    flush=True
)


NOVA WINNER: TREE_60_LINEAR40 CV ROC-AUC: 0.745043

NOVA GENERALIZATION CHECKS:
LATER DISCHARGES: train=0.77705 holdout=0.73365 gap=0.04340 n=15395
UNSEEN HOSPITALS: train=0.77148 holdout=0.72722 gap=0.04426 n=9407
SAVED: /kaggle/working/NOVA_BEST.csv candidate: TREE_60_LINEAR40


SAVED: /kaggle/working/NOVA_TREE_ONLY.csv candidate: TREE_SMALL



Historical V9 OOF: 0.748383; public: 0.74214. Review results before submitting.


In [6]:

import numpy as np
import pandas as pd
import re

needed = [
    'NOVA_X2', 'NOVA_T2', 'NOVA_y',
    'NOVA_ids', 'NOVA_n', 'NOVA_admit',
    'NOVA_discharge', 'nova_read', 'nova_dt'
]

assert all(k in globals() for k in needed), (
    "Run NOVA Cells 1 and 2 first."
)

NB_all = pd.concat(
    [NOVA_X2, NOVA_T2],
    ignore_index=True
).copy()

NB_added = []

def nb_add(frame, label):
    global NB_all

    if frame is None or frame.empty:
        return

    frame = frame.copy()
    frame.index = frame.index.astype(str)

    frame = frame[
        ~frame.index.duplicated(keep='first')
    ]

    frame = frame.reindex(
        NOVA_ids
    ).reset_index(drop=True)

    frame.columns = [
        f'nb_{label}_{i}'
        for i in range(frame.shape[1])
    ]

    NB_all = pd.concat(
        [NB_all, frame],
        axis=1
    )

    NB_added.append((
        label,
        len(frame.columns),
        int(frame.notna().any(axis=1).sum())
    ))

    print(
        'ADDED', label,
        frame.shape[1], 'features',
        flush=True
    )


# 1. Exact diagnosis codes, not only broad groups

dx = nova_read(
    'diagnoses.csv',
    {
        'encounter_id',
        'icd10_code',
        'icd_10_code',
        'diagnosis_code',
        'diagnosis_type',
        'is_primary'
    }
)

if dx is not None and 'encounter_id' in dx:

    codecol = next(
        (
            c for c in [
                'icd10_code',
                'icd_10_code',
                'diagnosis_code'
            ]
            if c in dx
        ),
        None
    )

    if codecol:
        dx = dx.dropna(
            subset=['encounter_id']
        ).copy()

        dx['encounter_id'] = (
            dx.encounter_id.astype(str)
        )

        dx['_code'] = (
            dx[codecol]
            .fillna('UNKNOWN')
            .astype(str)
            .str.upper()
            .str.replace(
                r'[^A-Z0-9]', '',
                regex=True
            )
        )

        keep = (
            dx['_code']
            .value_counts()
            .head(140)
            .index
        )

        subset = dx[
            dx['_code'].isin(keep)
        ]

        counts = pd.crosstab(
            subset.encounter_id,
            subset['_code']
        ).astype('float32')

        nb_add(
            counts,
            'exact_icd'
        )

        primary = pd.Series(
            False, index=dx.index
        )

        if 'is_primary' in dx:
            primary = (
                dx.is_primary
                .astype(str)
                .str.lower()
                .isin([
                    '1', 'true',
                    'yes', 'y'
                ])
            )

        elif 'diagnosis_type' in dx:
            primary = (
                dx.diagnosis_type
                .astype(str)
                .str.lower()
                .str.contains(
                    'primary|principal',
                    regex=True
                )
            )

        if primary.any():
            pp = dx.loc[primary].copy()

            pkeep = (
                pp['_code']
                .value_counts()
                .head(45)
                .index
            )

            pp = pp[
                pp['_code'].isin(pkeep)
            ]

            nb_add(
                pd.crosstab(
                    pp.encounter_id,
                    pp['_code']
                ).astype('float32'),
                'primary_icd'
            )

        del counts, subset

    del dx


# 2. Last 24-hour numerical laboratory measurements

lab = nova_read(
    'lab_results.csv',
    {
        'encounter_id',
        'collected_dt',
        'collection_dt',
        'result_dt',
        'lab_dt',
        'charttime',
        'lab_name',
        'test_name',
        'lab_test',
        'analyte',
        'value',
        'result_value',
        'numeric_value',
        'lab_value',
        'result'
    }
)

if lab is not None and 'encounter_id' in lab:

    tc = next(
        (
            c for c in [
                'collected_dt',
                'collection_dt',
                'result_dt',
                'lab_dt',
                'charttime'
            ]
            if c in lab
        ),
        None
    )

    nc = next(
        (
            c for c in [
                'lab_name',
                'test_name',
                'lab_test',
                'analyte'
            ]
            if c in lab
        ),
        None
    )

    vc = next(
        (
            c for c in [
                'value',
                'result_value',
                'numeric_value',
                'lab_value',
                'result'
            ]
            if c in lab
        ),
        None
    )

    if tc and nc and vc:

        lab['encounter_id'] = (
            lab.encounter_id.astype(str)
        )

        lab['_t'] = nova_dt(
            lab[tc]
        )

        lab['_end'] = (
            lab.encounter_id.map(
                NOVA_discharge
            )
        )

        lab['_start'] = (
            lab.encounter_id.map(
                NOVA_admit
            )
        )

        h = (
            lab['_end'] - lab['_t']
        ).dt.total_seconds() / 3600

        lab = lab[
            h.between(0, 24)
            & lab['_t'].ge(
                lab['_start']
            )
        ].copy()

        lab['_val'] = pd.to_numeric(
            lab[vc],
            errors='coerce'
        )

        lab['_test'] = (
            lab[nc]
            .fillna('UNKNOWN')
            .astype(str)
            .str.upper()
            .str.strip()
        )

        lab = lab[
            lab['_val'].notna()
        ]

        if len(lab):
            top = (
                lab['_test']
                .value_counts()
                .head(25)
                .index
            )

            lab = lab[
                lab['_test'].isin(top)
            ].sort_values('_t')

            latest = lab.drop_duplicates(
                ['encounter_id', '_test'],
                keep='last'
            )

            wide = latest.pivot(
                index='encounter_id',
                columns='_test',
                values='_val'
            )

            nb_add(
                wide.astype('float32'),
                'last24_lab_value'
            )

            print(
                'LAST-24H LAB RECORDS:',
                len(lab),
                flush=True
            )

    del lab


# 3. Clinical interactions

interactions = [
    (
        'nova_med_count',
        'nova_stay_days',
        'meds_per_day'
    ),
    (
        'nova_dx_count',
        'nova_stay_days',
        'dx_per_day'
    ),
    (
        'nova_lab_abnormal_count',
        'nova_lab_count',
        'lab_abnormal_rate2'
    ),
    (
        'nova_prior_30d',
        'nova_stay_days',
        'prior30_by_stay'
    ),
    (
        'nova_prior_90d',
        'nova_prior_365d',
        'recent_prior_fraction'
    ),
    (
        'nova_transfer_count',
        'nova_stay_days',
        'transfers_per_day'
    ),
    (
        'nova_icu_count',
        'nova_stay_days',
        'icu_per_day'
    ),
    (
        'nova_lab_last24_abnormal',
        'nova_lab_last24_count',
        'last24_abnormal_rate'
    )
]

for a, b, name in interactions:

    if a in NB_all and b in NB_all:

        x = pd.to_numeric(
            NB_all[a],
            errors='coerce'
        )

        z = pd.to_numeric(
            NB_all[b],
            errors='coerce'
        )

        NB_all[
            'nb_ratio_' + name
        ] = x / (
            1 + z.clip(lower=0)
        )


NB_all = NB_all.replace(
    [np.inf, -np.inf],
    np.nan
)

NB_X = (
    NB_all.iloc[:NOVA_n]
    .reset_index(drop=True)
)

NB_T = (
    NB_all.iloc[NOVA_n:]
    .reset_index(drop=True)
)

assert len(NB_X) == len(NOVA_y)
assert len(NB_T) == len(NOVA_T2)

print(
    '\nNOVA-BOOST FEATURES READY:',
    NB_X.shape,
    'TEST:', NB_T.shape,
    flush=True
)

print(
    'NEW SOURCE SUMMARY:',
    NB_added
)


READ: diagnoses.csv (451976, 3)
ADDED exact_icd 82 features
ADDED primary_icd 45 features
READ: lab_results.csv (1668783, 4)
ADDED last24_lab_value 13 features
LAST-24H LAB RECORDS: 409669

NOVA-BOOST FEATURES READY: (76975, 504) TEST: (32812, 504)
NEW SOURCE SUMMARY: [('exact_icd', 82, 109787), ('primary_icd', 45, 109033), ('last24_lab_value', 13, 107663)]


In [7]:

import gc
import base64
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)
from IPython.display import HTML, display

needed = [
    'NB_X', 'NB_T',
    'NOVA_y', 'NOVA_test_ids',
    'NOVA_oof', 'NOVA_pred',
    'NOVA_scores', 'nova_blend'
]

assert all(
    k in globals() for k in needed
), "Run NOVA Cells 1-2 and BOOST Cell A first."


# Prepare consistent numerical features

NB_raw = pd.concat(
    [NB_X, NB_T],
    ignore_index=True
)

NB_cats = NB_raw.select_dtypes(
    exclude='number'
).columns.tolist()

NB_raw = pd.get_dummies(
    NB_raw,
    columns=NB_cats,
    dummy_na=True,
    dtype=np.uint8
)

NB_raw = NB_raw.loc[
    :, ~NB_raw.columns.duplicated()
]

NB_raw = NB_raw.replace(
    [np.inf, -np.inf],
    np.nan
)

trainpart = NB_raw.iloc[
    :len(NOVA_y)
]

keep = (
    (trainpart.notna().sum() >= 80)
    &
    (trainpart.nunique(
        dropna=True
    ) > 1)
)

NB_raw = NB_raw.loc[:, keep]

NB_raw.columns = [
    f'nbf_{i}'
    for i in range(
        NB_raw.shape[1]
    )
]

NB_raw = NB_raw.astype(
    np.float32
)

NB_train = (
    NB_raw.iloc[:len(NOVA_y)]
    .reset_index(drop=True)
)

NB_test = (
    NB_raw.iloc[len(NOVA_y):]
    .reset_index(drop=True)
)

print(
    'BOOST TRAIN:', NB_train.shape,
    'TEST:', NB_test.shape,
    flush=True
)


# Same folds as the earlier models

folds = list(
    StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    ).split(
        NB_train, NOVA_y
    )
)

NB_oof = {}
NB_testpred = {}
NB_report = []


for model_name in [
    'XGBOOST',
    'LIGHTGBM'
]:

    NB_oof[model_name] = np.zeros(
        len(NOVA_y)
    )

    NB_testpred[model_name] = np.zeros(
        len(NB_test)
    )

    for k, (tr, va) in enumerate(
        folds, 1
    ):

        print(
            'NOVA-BOOST',
            model_name,
            'FOLD', k, '/3',
            flush=True
        )

        if model_name == 'XGBOOST':

            model = XGBClassifier(
                n_estimators=950,
                max_depth=3,
                learning_rate=0.035,
                min_child_weight=18,
                subsample=0.85,
                colsample_bytree=0.83,
                reg_lambda=12,
                reg_alpha=1,
                tree_method='hist',
                objective='binary:logistic',
                eval_metric='auc',
                early_stopping_rounds=75,
                n_jobs=4,
                random_state=300 + k
            )

            model.fit(
                NB_train.iloc[tr],
                NOVA_y[tr],
                eval_set=[
                    (
                        NB_train.iloc[va],
                        NOVA_y[va]
                    )
                ],
                verbose=False
            )

            best_iter = (
                model.best_iteration + 1
            )

        else:

            model = LGBMClassifier(
                n_estimators=1050,
                learning_rate=0.035,
                num_leaves=11,
                min_child_samples=100,
                colsample_bytree=0.83,
                subsample=0.85,
                subsample_freq=1,
                reg_lambda=14,
                reg_alpha=1,
                n_jobs=4,
                verbosity=-1,
                random_state=400 + k
            )

            model.fit(
                NB_train.iloc[tr],
                NOVA_y[tr],
                eval_set=[
                    (
                        NB_train.iloc[va],
                        NOVA_y[va]
                    )
                ],
                eval_metric='auc',
                callbacks=[
                    early_stopping(
                        85, verbose=False
                    ),
                    log_evaluation(0)
                ]
            )

            best_iter = (
                model.best_iteration_
            )

        NB_oof[model_name][va] = (
            model.predict_proba(
                NB_train.iloc[va]
            )[:, 1]
        )

        NB_testpred[model_name] += (
            model.predict_proba(
                NB_test
            )[:, 1] / 3
        )

        train_auc = roc_auc_score(
            NOVA_y[tr],
            model.predict_proba(
                NB_train.iloc[tr]
            )[:, 1]
        )

        val_auc = roc_auc_score(
            NOVA_y[va],
            NB_oof[model_name][va]
        )

        NB_report.append((
            model_name,
            k,
            train_auc,
            val_auc,
            train_auc - val_auc,
            best_iter
        ))

        print(
            'VAL:', round(val_auc, 6),
            'GAP:',
            round(
                train_auc - val_auc,
                6
            ),
            'TREES:', best_iter,
            flush=True
        )

        del model
        gc.collect()


# Blend against the original NOVA winner

NB_oldname = NOVA_scores[0][2]

NB_old_oof = nova_blend(
    NB_oldname,
    NOVA_oof
)

NB_old_test = nova_blend(
    NB_oldname,
    NOVA_pred
)

NB_new_oof = (
    0.5 * NB_oof['XGBOOST']
    + 0.5 * NB_oof['LIGHTGBM']
)

NB_new_test = (
    0.5 * NB_testpred['XGBOOST']
    + 0.5 * NB_testpred['LIGHTGBM']
)


NB_candidates = {
    'NOVA_ORIGINAL': (
        NB_old_oof,
        NB_old_test
    ),

    'BOOST_XGBOOST': (
        NB_oof['XGBOOST'],
        NB_testpred['XGBOOST']
    ),

    'BOOST_LIGHTGBM': (
        NB_oof['LIGHTGBM'],
        NB_testpred['LIGHTGBM']
    ),

    'BOOST_NEW_50_50': (
        NB_new_oof,
        NB_new_test
    ),

    'BOOST_OLD20_NEW80': (
        0.2 * NB_old_oof
        + 0.8 * NB_new_oof,

        0.2 * NB_old_test
        + 0.8 * NB_new_test
    ),

    'BOOST_OLD50_NEW50': (
        0.5 * NB_old_oof
        + 0.5 * NB_new_oof,

        0.5 * NB_old_test
        + 0.5 * NB_new_test
    ),

    'BOOST_OLD70_XGB30': (
        0.7 * NB_old_oof
        + 0.3 * NB_oof['XGBOOST'],

        0.7 * NB_old_test
        + 0.3 * NB_testpred['XGBOOST']
    )
}


NB_ranking = sorted(
    [
        (
            roc_auc_score(
                NOVA_y,
                p[0]
            ),
            name
        )
        for name, p in NB_candidates.items()
    ],
    reverse=True
)


print(
    '\nNOVA-BOOST OOF COMPARISON:'
)

for auc, name in NB_ranking:

    fold_gains = [
        roc_auc_score(
            NOVA_y[va],
            NB_candidates[name][0][va]
        )
        -
        roc_auc_score(
            NOVA_y[va],
            NB_old_oof[va]
        )
        for tr, va in folds
    ]

    print(
        name,
        'AUC=', round(auc, 6),
        'FOLD GAINS=',
        np.round(
            fold_gains, 6
        ),
        flush=True
    )


print(
    '\nTRAINING VS VALIDATION GAP'
)

print(
    pd.DataFrame(
        NB_report,
        columns=[
            'model',
            'fold',
            'train_auc',
            'val_auc',
            'gap',
            'trees'
        ]
    ).round(5).to_string(
        index=False
    )
)

print(
    '\nV9 HISTORICAL OFFLINE: 0.748383'
)

print(
    'V9 PUBLIC: 0.74214'
)


best_auc, best_name = (
    NB_ranking[0]
)

if best_auc < 0.748383:
    print(
        '\nWARNING: Still below V9 offline.',
        'Do not submit automatically.'
    )
else:
    print(
        '\nAbove V9 offline.',
        'Review fold consistency before submission.'
    )


# Save both candidates

for filename, name in [
    (
        'NOVA_BOOST_BEST.csv',
        best_name
    ),
    (
        'NOVA_BOOST_XGB.csv',
        'BOOST_XGBOOST'
    )
]:

    pred = np.asarray(
        NB_candidates[name][1]
    )

    assert len(pred) == len(
        NOVA_test_ids
    )

    assert np.isfinite(
        pred
    ).all()

    assert (
        (pred >= 0) & (pred <= 1)
    ).all()

    out = pd.DataFrame({
        'encounter_id':
            NOVA_test_ids.to_numpy(),

        'readmit_30d': pred
    })

    assert out.encounter_id.is_unique

    path = (
        '/kaggle/working/'
        + filename
    )

    out.to_csv(
        path,
        index=False
    )

    with open(path, 'rb') as f:
        encoded = base64.b64encode(
            f.read()
        ).decode('ascii')

    display(
        HTML(
            '<a download="'
            + filename
            + '" href="data:text/csv;base64,'
            + encoded
            + '">DOWNLOAD '
            + filename
            + '</a>'
        )
    )

    print(
        'SAVED:', path,
        'MODEL:', name
    )


BOOST TRAIN: (76975, 610) TEST: (32812, 610)
NOVA-BOOST XGBOOST FOLD 1 /3
VAL: 0.746285 GAP: 0.041243 TREES: 495
NOVA-BOOST XGBOOST FOLD 2 /3
VAL: 0.743159 GAP: 0.046583 TREES: 555
NOVA-BOOST XGBOOST FOLD 3 /3
VAL: 0.735068 GAP: 0.057768 TREES: 548
NOVA-BOOST LIGHTGBM FOLD 1 /3
VAL: 0.745403 GAP: 0.053786 TREES: 361
NOVA-BOOST LIGHTGBM FOLD 2 /3
VAL: 0.743278 GAP: 0.055054 TREES: 366
NOVA-BOOST LIGHTGBM FOLD 3 /3
VAL: 0.73595 GAP: 0.066544 TREES: 365

NOVA-BOOST OOF COMPARISON:
BOOST_OLD70_XGB30 AUC= 0.745282 FOLD GAINS= [ 0.000755 -0.000336  0.000282]
NOVA_ORIGINAL AUC= 0.745043 FOLD GAINS= [0. 0. 0.]
BOOST_OLD50_NEW50 AUC= 0.744979 FOLD GAINS= [ 0.000656 -0.001008  0.000138]
BOOST_OLD20_NEW80 AUC= 0.743532 FOLD GAINS= [-0.000453 -0.002934 -0.001162]
BOOST_NEW_50_50 AUC= 0.741919 FOLD GAINS= [-0.001895 -0.004799 -0.002694]
BOOST_LIGHTGBM AUC= 0.741459 FOLD GAINS= [-0.002882 -0.005164 -0.002754]
BOOST_XGBOOST AUC= 0.741384 FOLD GAINS= [-0.002    -0.005282 -0.003636]

TRAINING VS VALIDA

SAVED: /kaggle/working/NOVA_BOOST_BEST.csv MODEL: BOOST_OLD70_XGB30


SAVED: /kaggle/working/NOVA_BOOST_XGB.csv MODEL: BOOST_XGBOOST
